In [50]:
import pandas as pd
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
import re
from nltk.stem import WordNetLemmatizer

In [51]:
df= pd.read_csv('/kaggle/input/datasets/hassanamin/textdb3/fake_or_real_news.csv')

In [52]:
df.describe()

,Unnamed: 0
count,6335.000000
mean,5280.415627
std,3038.503953
min,2.000000
25%,2674.500000
50%,5271.000000
75%,7901.000000
max,10557.000000


In [53]:
df.head(10)

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL
5,6903,"Tehran, USA","\nI’m not an immigrant, but my grandparents ...",FAKE
6,7341,Girl Horrified At What She Watches Boyfriend D...,"Share This Baylee Luciani (left), Screenshot o...",FAKE
7,95,‘Britain’s Schindler’ Dies at 106,A Czech stockbroker who saved more than 650 Je...,REAL
8,4869,Fact check: Trump and Clinton at the 'commande...,Hillary Clinton and Donald Trump made some ina...,REAL
9,2909,Iran reportedly makes new push for uranium con...,Iranian negotiators reportedly have made a las...,REAL


In [54]:
df.columns

Index(['Unnamed: 0', 'title', 'text', 'label'], dtype='object')

In [55]:
df.isnull().sum()

Unnamed: 0    0
title         0
text          0
label         0
dtype: int64

In [56]:
df=df.drop(['Unnamed: 0', 'title'], axis=1)

In [57]:
df

,text,label
0,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,It's primary day in New York and front-runners...,REAL
...,...,...
6330,The State Department told the Republican Natio...,REAL
6331,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,FAKE
6332,Anti-Trump Protesters Are Tools of the Oligar...,FAKE
6333,"ADDIS ABABA, Ethiopia —President Obama convene...",REAL


In [58]:
df["label"]= df["label"].map({
    "FAKE":0,
    "REAL":1
})

In [59]:
df

,text,label
0,"Daniel Greenfield, a Shillman Journalism Fello...",0
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,0
2,U.S. Secretary of State John F. Kerry said Mon...,1
3,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",0
4,It's primary day in New York and front-runners...,1
...,...,...
6330,The State Department told the Republican Natio...,1
6331,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,0
6332,Anti-Trump Protesters Are Tools of the Oligar...,0
6333,"ADDIS ABABA, Ethiopia —President Obama convene...",1


In [62]:
snow_stemmer=SnowballStemmer('english')
lemmatizer=WordNetLemmatizer()

In [63]:
def stemming(content):
    con= re.sub('[^a-zA-Z]', ' ', content)
    con=con.lower()
    con=con.split()
    con=[lemmatizer.lemmatize(word) for word in con if not word in stopwords.words('english') ]
    con=' '.join(con)
    return con

In [64]:
df["text"]=df["text"].apply(stemming)

In [65]:
df

,text,label
0,daniel greenfield shillman journalism fellow f...,0
1,google pinterest digg linkedin reddit stumbleu...,0
2,u secretary state john f kerry said monday sto...,1
3,kaydee king kaydeeking november lesson tonight...,0
4,primary day new york front runner hillary clin...,1
...,...,...
6330,state department told republican national comm...,1
6331,p pb stand plutocratic pentagon posted oct wik...,0
6332,anti trump protester tool oligarchy reform alw...,0
6333,addis ababa ethiopia president obama convened ...,1


In [85]:
x=df["text"]

In [87]:
y=df["label"]

In [94]:
from sklearn.model_selection import train_test_split

In [99]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2)

In [101]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [102]:
vect=TfidfVectorizer()

In [103]:
x_train=vect.fit_transform(x_train)
x_test=vect.transform(x_test)

In [104]:
x_test.shape

(1267, 52896)

In [131]:
from sklearn.tree import DecisionTreeClassifier
model_dt=DecisionTreeClassifier(random_state=10)
model_dt.fit(x_train,y_train)
prediction_dt=model_dt.predict(x_test)
model_dt.score(x_test,y_test)

0.7924230465666929

In [132]:
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(random_state=10)
model_rf.fit(x_train, y_train)
prediction_rf = model_rf.predict(x_test)
model_rf.score(x_test, y_test)

0.8902920284135754

In [133]:
from sklearn.linear_model import LogisticRegression
model_lr = LogisticRegression(random_state=10)
model_lr.fit(x_train, y_train)
prediction_lr = model_lr.predict(x_test)
model_lr.score(x_test, y_test)

0.9179163378058406

In [134]:
from sklearn.ensemble import GradientBoostingClassifier
model_gb = GradientBoostingClassifier(random_state=10)
model_gb.fit(x_train, y_train)
prediction_gb = model_gb.predict(x_test)
print(model_gb.score(x_test, y_test))

0.8760852407261247


In [135]:
from xgboost import XGBClassifier
model_xgb = XGBClassifier(random_state=10)
model_xgb.fit(x_train, y_train)
prediction_xgb = model_xgb.predict(x_test)
print(model_xgb.score(x_test, y_test))

0.9108129439621152
